# Week 1 Day 3-4: 手写 Multi-Head Attention

**目标**: 不依赖 `nn.MultiheadAttention`,从零实现 scaled dot-product attention,加上 causal mask 和 multi-head 拆分。

**视频对照**:
- Karpathy "Let's build GPT": https://www.youtube.com/watch?v=kCc8FmEb1nY

**验收点**:
- [ ] 在白板上画出 `(B,T,D) → (B,H,T,Dk) → attention → (B,T,D)` 的完整 shape 流
- [ ] 能解释 causal mask 为什么是上三角
- [ ] 能解释 `/sqrt(d_k)` 缩放的意义
- [ ] 一致性测试通过 (max abs diff < 1e-5)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## 手写实现

输入: `x` shape = `(B, T, D_model)`

输出: `y` shape = `(B, T, D_model)`

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    内部维度变换:
        x → W_q/W_k/W_v → q/k/v   (B, T, D_model)
        reshape + transpose         (B, H, T, D_k)   D_k = D_model / H
        attention(q, k, v)          (B, H, T, D_k)
        merge heads                 (B, T, D_model)
        W_o                         (B, T, D_model)
    """

    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 必须能整除 n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        
        # TODO 1: 投影 + reshape 到 (B, H, T, D_k)
        # 提示:
        #   q = self.W_q(x)                 # (B, T, D)
        #   q = q.view(B, T, H, Dk).transpose(1, 2)   # (B, H, T, Dk)
        # k, v 同理
        # q, k, v = ?, ?, ?
        
        # TODO 2: 算 attention scores
        # scores = q @ k^T / sqrt(d_k)     # (B, H, T, T)
        # scores = ?
        
        # TODO 3: 加 causal mask
        # mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        # scores = scores.masked_fill(mask, float('-inf'))
        
        # TODO 4: softmax + 加权求和
        # attn = F.softmax(scores, dim=-1)
        # out  = attn @ v                   # (B, H, T, D_k)
        
        # TODO 5: 合并 heads + 输出投影
        # out = out.transpose(1, 2).contiguous().view(B, T, D)
        # return self.W_o(out)
        
        raise NotImplementedError("按 TODO 提示实现 5 步")

# 测试模块能否实例化
mha = MultiHeadAttention(d_model=64, n_heads=4)
print(f"模块创建成功: {mha}")

## 用 PyTorch SDPA 作为参考验证

`F.scaled_dot_product_attention` 是 PyTorch 内置的高效实现,用作 ground truth。

In [ ]:
def reference_attention(q, k, v):
    """输入 q,k,v 都是 (B, H, T, D_k),输出 (B, H, T, D_k)"""
    return F.scaled_dot_product_attention(q, k, v, is_causal=True)

# 测试: 用同样的 q,k,v 对比手写公式和 SDPA
torch.manual_seed(0)
B, T, D, H = 2, 5, 32, 4
Dk = D // H

x = torch.randn(B, T, D)
W_q = nn.Linear(D, D, bias=False)
W_k = nn.Linear(D, D, bias=False)
W_v = nn.Linear(D, D, bias=False)

def proj(W, x):
    return W(x).view(B, T, H, Dk).transpose(1, 2)

q, k, v = proj(W_q, x), proj(W_k, x), proj(W_v, x)

# 手写版
scale = 1.0 / math.sqrt(Dk)
scores = (q @ k.transpose(-2, -1)) * scale
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))
attn = F.softmax(scores, dim=-1)
out_manual = attn @ v

# 参考版
out_ref = reference_attention(q, k, v)

diff = (out_manual - out_ref).abs().max().item()
ok = "✅" if diff < 1e-5 else "❌"
print(f"manual vs SDPA max abs diff = {diff:.2e}  {ok}")

## 可视化 attention scores

看看 causal mask 长什么样

In [ ]:
import matplotlib.pyplot as plt

T = 10
scores = torch.randn(1, 1, T, T)
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
scores_masked = scores.masked_fill(mask, float('-inf'))
attn = F.softmax(scores_masked, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(mask[0,0].float(), cmap='gray')
axes[0].set_title("Causal Mask (白色=被屏蔽)")
axes[1].imshow(attn[0,0].detach(), cmap='viridis')
axes[1].set_title("Softmax 后的 Attention 分布")
plt.tight_layout()
plt.show()

---

**交付物**:
- 填完 TODO 后的 `MultiHeadAttention` 完整实现
- 白板照片: Q/K/V shape 流 → `phase0/notes/week1/`
- 一致性测试通过截图